# Day 3 · Hands-on 2 — Experimental design: pseudoreplication, batches, blocking and randomisation
**About 40 min · work in pairs** (one drives, one reads, predicts and questions — swap halfway)

No analysis can rescue a badly designed experiment. This notebook uses **simulated experiments**, so we always know the truth, and repeats each one thousands of times to see how often each analysis gets it right. Four questions:

- **A. Pseudoreplication:** what is my sample size, really?
- **B. Batch effects:** what happens when treatment and batch line up?
- **C. Blocking:** how pairing on a known source of variation buys power for free.
- **D. Randomisation:** why "I just picked the first mice I caught" biases results.

> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to write, change, fix or explain *code*, including new simulations.
> - 🎓 **Claude / ChatGPT (other tab) = tutor.** Use it to explain ideas, quiz you and critique your own work. Ask for its reasoning, and check one claim yourself in the notebook.

**How this notebook works:** run cells from top to bottom. Each code cell has one line marked `# 🔧 CHANGE THIS` — the only thing you need to edit. Lines starting with 📋 go in the shared sheet. Lines starting with 🎯 compare an estimate with the truth used to simulate it.

In [ ]:
# @title ▶️ Setup: run this cell first (no changes needed) { display-mode: "form" }
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
warnings.filterwarnings("ignore")

N_SIM = 2000          # simulated experiments per scenario

def ols_p(X, y, col=1):
    """p-value for coefficient `col` in a least-squares fit of y on design matrix X."""
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    df = len(y) - X.shape[1]
    s2 = resid @ resid / df
    se = np.sqrt(s2 * np.linalg.inv(X.T @ X)[col, col])
    return beta[col], 2 * stats.t.sf(abs(beta[col] / se), df)

def strip(ax, x, y, colors, jitter=0.08, rng=np.random.default_rng(0), **kw):
    ax.scatter(x + rng.uniform(-jitter, jitter, len(x)), y, c=colors, **kw)

print("✅ Setup complete")

## A. Pseudoreplication: what is *n*?
### 🧭 Concept
You treat **3 mice** and keep **3 control mice**, then image **50 cells per mouse**. Is n = 3 or n = 150 per group?

The treatment was applied to **mice**, so the mouse is the **experimental unit**. Cells are **observational units**: measuring more cells tells you more precisely about *that mouse*, but adds no new information about how *mice* respond to treatment. Mice differ from each other (genetics, cage, stress), and that between-mouse variation is what the test must compare the treatment effect against.

In this simulation the treatment has **no effect at all**.

✍️ **Predict first:** a t-test on all 150 vs 150 cells, and a t-test on the 3 vs 3 mouse averages. Which one is more likely to give p < 0.05?

In [ ]:
# 🔧 CHANGE THIS: your pair number (gives you a different random experiment)
seed = 1

mice_per_group, cells_per_mouse = 3, 50
mouse_sd, cell_sd = 1.0, 1.0           # spread between mice, and between cells of the same mouse
rng = np.random.default_rng(seed)
rows = []
for g in ["control", "treated"]:
    for m in range(mice_per_group):
        mouse_level = rng.normal(0, mouse_sd)          # no treatment effect: same distribution in both groups
        y = mouse_level + rng.normal(0, cell_sd, cells_per_mouse)
        rows.append(pd.DataFrame({"group": g, "mouse": f"{g[0].upper()}{m + 1}", "y": y}))
cells_df = pd.concat(rows, ignore_index=True)
mice_df = cells_df.groupby(["group", "mouse"], as_index=False)["y"].mean()

fig, ax = plt.subplots(figsize=(7, 4))
palette = plt.cm.tab10.colors
for i, (mouse, sub) in enumerate(cells_df.groupby("mouse", sort=False)):
    xpos = (0 if sub.group.iloc[0] == "control" else 1) + (i % mice_per_group - 1) * 0.22
    strip(ax, np.full(len(sub), xpos), sub.y, [palette[i]] * len(sub), jitter=0.06, s=8, alpha=0.5)
    ax.plot(xpos, sub.y.mean(), "D", color=palette[i], markeredgecolor="black", markersize=10)
ax.set_xticks([0, 1], ["control", "treated"]); ax.set_ylabel("Measurement")
ax.set_title("Dots = cells, diamonds = mouse averages (colour = mouse)"); plt.show()

p_cells = stats.ttest_ind(*[cells_df.y[cells_df.group == g] for g in ["control", "treated"]]).pvalue
p_mice = stats.ttest_ind(*[mice_df.y[mice_df.group == g] for g in ["control", "treated"]]).pvalue
mixed = smf.mixedlm("y ~ group", cells_df, groups=cells_df["mouse"]).fit()
print(f"t-test on cells        (n = {cells_per_mouse * mice_per_group} per group): p = {p_cells:.2g}")
print(f"t-test on mouse means  (n = {mice_per_group} per group):   p = {p_mice:.2g}")
print(f"Mixed model (cells nested in mice):        p = {mixed.pvalues['group[T.treated]']:.2g}")
print(f"📋 seed {seed}: p cells = {p_cells:.2g}, p mice = {p_mice:.2g}")

Your single experiment may or may not have been "significant". To see how often each analysis is fooled, repeat the whole experiment 2,000 times. Because the treatment does nothing, a correct analysis should give p < 0.05 in about **5%** of experiments.

✍️ **Predict first:** if you measure **more cells per mouse**, does the cell-level false-positive rate go down, stay the same, or go up?

In [ ]:
# 🔧 CHANGE THIS: cells measured per mouse (try 5, 20, 50, 200)
cells_per_mouse = 50

rng = np.random.default_rng(2026)
mouse_level = rng.normal(0, mouse_sd, (N_SIM, 2, mice_per_group, 1))
y = mouse_level + rng.normal(0, cell_sd, (N_SIM, 2, mice_per_group, cells_per_mouse))
flat = y.reshape(N_SIM, 2, -1)
p_cells = stats.ttest_ind(flat[:, 0], flat[:, 1], axis=1).pvalue
means = y.mean(axis=3)
p_mice = stats.ttest_ind(means[:, 0], means[:, 1], axis=1).pvalue

icc = mouse_sd**2 / (mouse_sd**2 + cell_sd**2)
deff = 1 + (cells_per_mouse - 1) * icc
print(f"False-positive rate, t-test on cells:       {np.mean(p_cells < 0.05):.1%}")
print(f"False-positive rate, t-test on mouse means: {np.mean(p_mice < 0.05):.1%}   (should be about 5%)")
print(f"\nIntraclass correlation (ICC) = {icc:.2f}: cells from the same mouse are alike")
print(f"Effective sample size per group ≈ {cells_per_mouse * mice_per_group} cells / {deff:.1f} = {cells_per_mouse * mice_per_group / deff:.1f}  (you only have {mice_per_group} mice!)")
print(f"📋 cells_per_mouse = {cells_per_mouse}: false positives cells = {np.mean(p_cells < 0.05):.0%}, mice = {np.mean(p_mice < 0.05):.0%}")

### ✅ Concept check
Why does the cell-level false-positive rate get **worse** as you measure more cells?

<details><summary>Show answer</summary>

The cell-level t-test thinks its standard error shrinks with √(number of cells). But the real uncertainty is dominated by which 3 mice you happened to get, and that doesn't shrink at all when you measure more cells. More cells make the test more confident about a difference that is really just mouse-to-mouse variation.

Valid options: (1) average per mouse and analyse the mouse means (simple, and exactly right when every mouse has the same number of cells); (2) a **mixed model** with mouse as a random effect, which also uses the cell-level spread and handles unequal cell numbers. Both say n = 3. To gain power, add **mice**, not cells.
</details>

Pseudoreplication hides everywhere: wells from the same plate, technical replicates of the same RNA, repeated measurements on the same patient, fish in the same tank (if the drug went in the tank water, the **tank** is the experimental unit).

## B. Batch effects and confounding
### 🧭 Concept
You have 12 samples: 6 control, 6 treated. Only 6 fit in one run (sequencing lane, plate, staining day), so you process them in **two batches**. Batches always differ a little (reagent lot, temperature, operator). Here batch B reads **1.5 units higher** than batch A, and the treatment again has **no effect**.

Two ways to assign samples to batches:
- **confounded:** all controls in batch A, all treated in batch B (the convenient choice: "do the controls first")
- **balanced:** 3 controls + 3 treated in each batch

✍️ **Predict first:** in the confounded design, can adding `batch` to the model rescue the analysis?

In [ ]:
# 🔧 CHANGE THIS: "confounded" or "balanced"
design = "confounded"

batch_shift, noise_sd = 1.5, 1.0
treatment = np.repeat([0, 1], 6)                                       # 6 control, 6 treated
batch = treatment.copy() if design == "confounded" else np.tile([0, 0, 0, 1, 1, 1], 2)
rng = np.random.default_rng(seed)
y = batch_shift * batch + rng.normal(0, noise_sd, 12)                  # no treatment effect
exp_df = pd.DataFrame({"treatment": np.where(treatment, "treated", "control"),
                       "batch": np.where(batch, "B", "A"), "y": y})
display(pd.crosstab(exp_df.treatment, exp_df.batch))

fig, ax = plt.subplots(figsize=(6, 3.5))
for b, color in [("A", "tab:blue"), ("B", "tab:orange")]:
    sub = exp_df[exp_df.batch == b]
    strip(ax, (sub.treatment == "treated").astype(float).to_numpy(), sub.y, [color] * len(sub), s=50, label=f"batch {b}")
ax.set_xticks([0, 1], ["control", "treated"]); ax.set_ylabel("Measurement"); ax.legend(); plt.show()

fit1 = smf.ols("y ~ treatment", exp_df).fit()
print(f"y ~ treatment:          effect = {fit1.params['treatment[T.treated]']:+.2f}, p = {fit1.pvalues['treatment[T.treated]']:.2g}")
X = np.column_stack([np.ones(12), treatment, batch])
if np.linalg.matrix_rank(X) < 3:
    print("y ~ treatment + batch:  cannot be estimated. Treatment and batch are the SAME column:")
    print("                        any difference could be treatment, batch, or any mix of the two.")
else:
    fit2 = smf.ols("y ~ treatment + batch", exp_df).fit()
    print(f"y ~ treatment + batch:  effect = {fit2.params['treatment[T.treated]']:+.2f}, p = {fit2.pvalues['treatment[T.treated]']:.2g}")
print("🎯 True treatment effect = 0; batch B is 1.5 higher than batch A")

Now repeat each design 2,000 times. Start with no treatment effect (false-positive rate), then give the treatment a real effect (power). The `design` you chose above carries over.

✍️ **Predict first:** in the balanced design, does adjusting for batch matter if batch is already balanced?

In [ ]:
# 🔧 CHANGE THIS: true treatment effect (0 = none; then try 1.5)
true_effect = 0.0

rng = np.random.default_rng(2026)
p_ignore, p_adjust, est = [], [], []
X1 = np.column_stack([np.ones(12), treatment])
X2 = np.column_stack([np.ones(12), treatment, batch])
for _ in range(N_SIM):
    y = true_effect * treatment + batch_shift * batch + rng.normal(0, noise_sd, 12)
    b, p = ols_p(X1, y); p_ignore.append(p); est.append(b)
    if design == "balanced":
        p_adjust.append(ols_p(X2, y)[1])
what = "false-positive rate" if true_effect == 0 else "power"
print(f"Design: {design}, true effect = {true_effect}")
print(f"Average estimated effect (ignoring batch): {np.mean(est):+.2f}   🎯 truth = {true_effect:+.2f}")
print(f"{what}, ignoring batch:  {np.mean(np.array(p_ignore) < 0.05):.0%}")
if design == "balanced":
    print(f"{what}, adjusting for batch: {np.mean(np.array(p_adjust) < 0.05):.0%}")
else:
    print(f"{what}, adjusting for batch: impossible (treatment and batch are confounded)")
print(f"📋 {design}, effect {true_effect}: {what} ignoring batch {np.mean(np.array(p_ignore) < 0.05):.0%}"
      + (f", adjusting {np.mean(np.array(p_adjust) < 0.05):.0%}" if p_adjust else ""))

### ✅ Concept check
A colleague's RNA-seq experiment has all controls sequenced in January and all treated samples in March. They plan to "remove the batch effect with ComBat". What happens?

<details><summary>Show answer</summary>

Batch-correction tools estimate the batch effect from differences between batches. When batch and treatment are perfectly confounded, the batch difference **is** the treatment difference, so the tool either refuses or removes the treatment effect along with the batch (or, if told to protect treatment, removes nothing). No method can separate them after the fact. The fix is in the design: spread every condition across every batch. In the balanced design, batch is not a bias at all; putting it in the model removes its noise from the residuals, which is why adjusting increased power.
</details>

## C. Blocking: compare like with like
### 🧭 Concept
Six donors each give cells, and you split each donor's cells into a control and a treated dish. Donors differ a lot from each other (`block_sd`), but within a donor the control and treated dishes share everything except the treatment. A **block** is any group of units that are more alike than units from different groups: a donor, a litter, a cell line, a plate, a day.

Analysing **within blocks** (a paired t-test, or a model with `+ donor`) removes the donor-to-donor variation from the comparison. On Day 1 (card 2) and Day 2 (airway, with the cell line as a block) you saw the payoff with real data.

The treatment here **does** have an effect: +1.0.

✍️ **Predict first:** as donors become more different from each other (bigger `block_sd`), what happens to the power of the unpaired test? And of the paired test?

In [ ]:
# 🔧 CHANGE THIS: how different the donors are from each other (try 0, 0.5, 1, 2)
block_sd = 1.0

n_donors, effect, within_sd = 6, 1.0, 0.5
rng = np.random.default_rng(seed)
donor = rng.normal(0, block_sd, n_donors)
ctrl = donor + rng.normal(0, within_sd, n_donors)
trt = donor + effect + rng.normal(0, within_sd, n_donors)

fig, ax = plt.subplots(figsize=(5, 4))
for c, t in zip(ctrl, trt):
    ax.plot([0, 1], [c, t], "o-", color="gray", alpha=0.7)
ax.set_xticks([0, 1], ["control", "treated"]); ax.set_xlim(-0.4, 1.4); ax.set_ylabel("Measurement")
ax.set_title("One experiment: each line is one donor"); plt.show()
print(f"This experiment: unpaired p = {stats.ttest_ind(trt, ctrl).pvalue:.2g},  paired p = {stats.ttest_rel(trt, ctrl).pvalue:.2g}")

donors = rng.normal(0, block_sd, (N_SIM, n_donors))
C = donors + rng.normal(0, within_sd, (N_SIM, n_donors))
Tr = donors + effect + rng.normal(0, within_sd, (N_SIM, n_donors))
pow_unpaired = np.mean(stats.ttest_ind(Tr, C, axis=1).pvalue < 0.05)
pow_paired = np.mean(stats.ttest_rel(Tr, C, axis=1).pvalue < 0.05)
print(f"\nPower over {N_SIM} experiments:  unpaired = {pow_unpaired:.0%},  paired (blocked) = {pow_paired:.0%}")
print(f"📋 block_sd = {block_sd}: power unpaired {pow_unpaired:.0%}, paired {pow_paired:.0%}")

### ✅ Concept check
With `block_sd = 0`, the paired test is slightly *less* powerful than the unpaired one. Why, and should you then skip blocking?

<details><summary>Show answer</summary>

The paired test spends degrees of freedom estimating the donor effects (5 df instead of 10). If donors don't actually differ, that costs a little power for nothing. But you rarely know in advance, the cost is small, and the gain when blocks do differ is large. Rule of thumb: **block what you can, randomise what you can't**. And once you have blocked in the design, keep the block in the analysis: analysing a paired design as unpaired throws the benefit away.
</details>

## D. Randomisation
### 🧭 Concept
You have 20 mice and will treat 10. Heavier mice happen to have higher outcome values, and the treatment does **nothing**. Three ways to allocate:

- **first_caught:** treat the first 10 mice you catch from the cage. Heavier, slower mice tend to be caught first.
- **random:** randomly pick 10 of the 20.
- **matched_pairs:** rank the mice by weight, pair neighbours, and randomly treat one mouse in each pair (blocking + randomisation), then analyse the pairs.

✍️ **Predict first:** which allocation gives a biased average effect? Which gives the most precise estimate?

In [ ]:
# 🔧 CHANGE THIS: "first_caught", "random" or "matched_pairs"
allocation = "first_caught"

rng = np.random.default_rng(2026)
est, pvals = [], []
for _ in range(N_SIM):
    weight = rng.normal(25, 3, 20)
    y = 0.3 * (weight - 25) + rng.normal(0, 1, 20)                    # no treatment effect
    if allocation == "first_caught":
        order = np.argsort(-(weight + rng.normal(0, 3, 20)))          # heavier mice tend to be caught first
        treated = np.zeros(20, bool); treated[order[:10]] = True
    elif allocation == "random":
        treated = np.zeros(20, bool); treated[rng.choice(20, 10, replace=False)] = True
    else:
        order = np.argsort(weight).reshape(10, 2)                      # pairs of similar weight
        flip = rng.integers(0, 2, 10)
        treated = np.zeros(20, bool); treated[order[np.arange(10), flip]] = True
    if allocation == "matched_pairs":
        diff = y[order[np.arange(10), flip]] - y[order[np.arange(10), 1 - flip]]
        est.append(diff.mean()); pvals.append(stats.ttest_1samp(diff, 0).pvalue)
    else:
        est.append(y[treated].mean() - y[~treated].mean()); pvals.append(stats.ttest_ind(y[treated], y[~treated]).pvalue)

est = np.array(est)
plt.figure(figsize=(6, 3.2))
plt.hist(est, bins=50, color="gray", edgecolor="white"); plt.axvline(0, color="red", label="truth = 0")
plt.axvline(est.mean(), color="black", ls="--", label=f"average estimate = {est.mean():+.2f}")
plt.xlim(-2.5, 2.5); plt.xlabel("Estimated treatment effect"); plt.legend(); plt.title(allocation); plt.show()
print(f"Average estimate = {est.mean():+.2f} (bias),  SD of estimates = {est.std():.2f} (precision)")
print(f"False-positive rate = {np.mean(np.array(pvals) < 0.05):.0%}   🎯 should be about 5%")
print(f"📋 {allocation}: bias {est.mean():+.2f}, SD {est.std():.2f}, false positives {np.mean(np.array(pvals) < 0.05):.0%}")

### ✅ Concept check
"I didn't randomise, but I checked that the groups have the same average weight, so it's fine." Is it?

<details><summary>Show answer</summary>

Not necessarily. Weight is the variable you *measured*. Haphazard allocation can also line up with things you didn't measure (activity, dominance, cage position, time of day you processed them). Randomisation balances **all** of these on average, measured or not, and makes the p-value mean what it says. Balance checks only cover what you thought of. (And in a randomised experiment, a baseline difference is by definition due to chance, so "testing" for it answers no useful question.)

Blinding is the companion to randomisation: whoever measures the outcome should not know which mice were treated.
</details>

## E. Plan your own experiment
Take one experiment from your own work (past or planned) and fill in this table with your partner. Copy it into a text cell or the shared sheet.

| Question | Your answer |
|---|---|
| What is the treatment / exposure, and **what is it applied to**? (= experimental unit) | |
| What do you actually measure? (= observational unit) | |
| How many **experimental units** per group? | |
| Known sources of variation you could **block** on (donor, litter, plate, day…) | |
| **Batches** (runs, plates, lanes, days): is every condition in every batch? | |
| How is allocation **randomised**? Who is **blinded**? | |
| Which analysis respects this structure? (mean per unit, paired test, `+ block`, mixed model) | |

## 🤖 Go deeper with AI
Three ways to keep learning about experimental design: now, if you finish early, or after the workshop.

**🎓 Explain and quiz me** (Claude / ChatGPT):
> *Explain experimental unit vs observational unit with three lab examples. Then describe 4 experiments to me, one at a time, and ask me what n is in each.*

**🧪 See it in a simulation** (🧮 Gemini in Colab, in a new cell at the end of this notebook):
> *Simulate 3 vs 3 mice with 50 cells each and no true effect, 1,000 times. Compare the false-positive rate of a t-test on cells, a t-test on mouse means and a mixed model.*

**🔬 Apply it to my work** (Claude / ChatGPT; replace the brackets with your own details):
> *Here is my planned experiment: [describe]. Act as a critical reviewer: identify the experimental unit, likely batches and blocks, and how to randomise. Ask me questions before answering.*

✔ **Check one claim yourself:** The t-test on cells should give far more than 5% false positives; the other two about 5%.

## 💬 Discuss with your partner
- In your lab, what is the most common "n" that is really pseudoreplicated (technical replicates, cells, wells, fields of view)?
- Section A used ICC = 0.5. Would cells from the same mouse be more or less alike than that in your system? How would that change the false-positive rate?
- Section B: what is the cheapest change to your usual workflow that would stop treatment and batch lining up?
- When is it impossible to randomise (e.g. sex, genotype, age), and what does that mean for how you interpret a difference? (Think back to confounding in HO1, section D.)